# 03 — Extraction de features (physiques + FWT/TFUG placeholder)


- **Physiques** : pics, positions, aires, largeurs (interprétables).
- **Avancées** : *placeholder* FWT quaternionique / TFUG (interfaces prévues, implémentation à brancher).


In [ ]:

import numpy as np, pandas as pd
from scipy.signal import find_peaks

def features_physics(df, prominence=0.01, distance=5, topk=5):
    x = df["wavenumber_cm1"].to_numpy()
    y = df["intensity"].to_numpy()
    peaks, props = find_peaks(y, prominence=prominence, distance=distance)
    feats = {"n_peaks": int(len(peaks)),
             "y_mean": float(np.mean(y)),
             "y_std": float(np.std(y)),
             "area": float(np.trapz(y,x))}
    order = np.argsort(props.get("prominences", np.zeros_like(peaks)))[::-1]
    peaks_sorted = peaks[order][:topk]
    for i,p in enumerate(peaks_sorted):
        feats[f"peak_{i+1}_pos"] = float(x[p])
        feats[f"peak_{i+1}_height"] = float(y[p])
    return pd.DataFrame([feats])

# Placeholder interface FWT/TFUG
class TFUGFWTEmbedder:
    def __init__(self, level=4, basis="q-fwt-v1"):
        self.level = level; self.basis = basis
    def transform(self, df):
        x = df["wavenumber_cm1"].to_numpy()
        y = df["intensity"].to_numpy()
        # TODO: remplacer par vraie transformée FWT quaternionique
        # Embedding jouet = downsample + stats
        emb = np.concatenate([y[::max(1,len(y)//64)][:64],
                              [y.mean(), y.std()]])
        return emb

# Démo rapide
def demo_spectrum(n=2048, seed=1):
    rng = np.random.default_rng(seed); import numpy as np
    x = np.linspace(100, 3500, n); y = 0.05*rng.standard_normal(n)
    for c in [520, 1000, 1580]:
        y += np.exp(-0.5*((x-c)/5)**2)*(1.0 + 0.1*rng.normal(size=n))
    return pd.DataFrame({"sample_id":["demo"]*n,"wavenumber_cm1":x,"intensity":y})

df = demo_spectrum()
from scipy.signal import savgol_filter
df["intensity"] = savgol_filter(df["intensity"], 21, 3)
feats = features_physics(df); feats
